In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

# training process
import algorithm.algo2 as algo2
reload(algo2)

# evaluation
import algorithm.kmeans as kmeans
reload(kmeans)
import eval_metrics.ac

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args = parser.parse_args()

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

In [54]:
CLUSTER_NUM = 20
LAMBDA = 1

# CLUSTER_NUM = 2, LAMBDA = 2, [4,20], 100 iter -> 100%
# [ 8 11] 80 iter -> 87.5%

# k=4, lambda=1, [ 5 10 12 17], 100 iter -> 94.4%
# k=6, lambda=1, [ 7 11 13 15 16 18], 100 iter -> 95.1%
# k=10, lambda=0.3, [ 2  4  5  7  8  9 15 17 18 20], 500 iter ->72.7%
# k=10, lambda=1, [ 1  2  4  6  8 11 12 13 15 16], 800 iter -> 0.81

In [55]:
# read data
mat, label = coil20.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)
mat -= torch.mean(mat)

print(f'[Using classes] {np.unique(np.array(label))}')

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T
A.to(device)
pass

[Using classes] [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]


In [56]:
reload(algo2)
H = torch.randn(A.shape[0], CLUSTER_NUM)
H.to(device)

H_out = algo2.matrixFac(A=A, H=H, 
                        lamda=LAMBDA, 
                        num_iter=3000,
                        print_every=100)

iter 100 done, loss = 7180.1162109375, step = 0.0007392537663690746
iter 200 done, loss = 2528.28955078125, step = 0.0007549584843218327
iter 300 done, loss = 1310.95458984375, step = 0.0007618374656885862
iter 400 done, loss = 852.1881713867188, step = 0.0007653599022887647
iter 500 done, loss = 634.6942138671875, step = 0.0007674527587369084
iter 600 done, loss = 514.4585571289062, step = 0.0007687818142585456
iter 700 done, loss = 441.4148254394531, step = 0.000769674894399941
iter 800 done, loss = 394.65216064453125, step = 0.0007702884031459689
iter 900 done, loss = 363.6057434082031, step = 0.0007707164622843266
iter 1000 done, loss = 343.881103515625, step = 0.0007709958590567112
iter 1100 done, loss = 331.82598876953125, step = 0.000771168852224946
iter 1200 done, loss = 324.65875244140625, step = 0.0007712719379924238
iter 1300 done, loss = 320.3088684082031, step = 0.0007713348022662103
iter 1400 done, loss = 317.4556579589844, step = 0.000771376711782068
iter 1500 done, loss

In [57]:
reload(kmeans)
pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM, max_iter=1000)
acc = eval_metrics.ac.ac(label, pred_labels)
print(f'[Accuracy] {acc}')

/workspace/MSBD5002-Project/implementation/experiment/3_more_generalized_algorithm/../../algorithm/kmeans.py:45: RuntimeWarning: Mean of empty slice.
  centers[i] = X[labels == i].mean(axis=0)
/workspace/anaconda3/envs/5001Proj/lib/python3.9/site-packages/numpy/core/_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


K-means algorithm did not converge in 1000 iterations
K-means algorithm did not converge in 1000 iterations
[Accuracy] 0.6430555555555556
